# 1D Channel Capacity Analysis

This notebook demonstrates the `HdfChannelCapacity` class for analyzing 1D channel capacity
using the HCFCD (Harris County Flood Control District) methodology. Channel capacity is
determined by comparing water surface elevations from multiple AEP storm profiles against
bank elevations at each cross section.

**Capacity Levels:**

| Level | Category | Description |
|-------|----------|-------------|
| 1 | X ≤ 10% AEP | Only contains 10-year storm or smaller |
| 2 | 10% < X ≤ 4% | Contains up to 25-year storm |
| 3 | 4% < X ≤ 2% | Contains up to 50-year storm |
| 4 | 2% < X ≤ 1% | Contains up to 100-year storm |
| 5 | 1% < X ≤ 0.2% | Contains up to 500-year storm |
| 6 | X > 0.2% | Contains all tested storms |
| 7 | All overtop | Overtopped by smallest tested storm |

**What you'll learn:**
- Extract bank elevations from geometry HDF
- Extract individual steady-state profile WSE from a single plan HDF
- Determine channel capacity level at each cross section
- Aggregate capacity into channel segments
- Generate system-wide capacity summaries
- Compare existing vs proposed conditions

**Data Source:** FEMA BLE model (Lower Colorado-Cummins, HUC 12090301), SHILOH BRANCH reach
with 40 cross sections and 7 steady-state flow profiles.

In [ ]:
# =============================================================================
# DEVELOPMENT MODE TOGGLE
# =============================================================================
USE_LOCAL_SOURCE = True  # <-- TOGGLE THIS

if USE_LOCAL_SOURCE:
    import sys
    from pathlib import Path
    local_path = str(Path.cwd().parent)
    if local_path not in sys.path:
        sys.path.insert(0, local_path)
    print(f"LOCAL SOURCE MODE: Loading from {local_path}/ras_commander")
else:
    print("PIP PACKAGE MODE: Loading installed ras-commander")

from ras_commander import init_ras_project, RasCmdr, RasExamples
from ras_commander.hdf.HdfChannelCapacity import (
    HdfChannelCapacity, STORM_ORDER, LEVEL_TO_CATEGORY
)
from ras_commander.sources.federal.ebfe_models import RasEbfeModels
import ras_commander
print(f"Loaded: {ras_commander.__file__}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## Setup: BLE Model Preparation

We use a FEMA Base Level Engineering (BLE) reach model — SHILOH BRANCH from the
Lower Colorado-Cummins study (HUC 12090301). BLE models are steady-state with 7
flow profiles representing different AEP events.

The `RasEbfeModels.organize_lower_colorado_cummins()` function handles downloading
the 290.7 MB BLE archive from S3 and organizing the selected reach into a standard
folder structure. Since BLE models were built with HEC-RAS 4.x, we re-run with
HEC-RAS 6.5 to generate HDF output files needed for analysis.

In [ ]:
# =============================================================================
# Parameters
# =============================================================================

# Reach to extract from BLE archive (Lower Colorado-Cummins, HUC 12090301)
BLE_RIVER = "Rabbs Creek-Colorado River"
BLE_REACH_NAME = "SHILOH BRANCH"

# HEC-RAS version for re-running (generates HDF)
RAS_VERSION = "6.5"

# BLE profile name to standard AEP mapping
# Only 5 of the 7 profiles are standard AEP events;
# '1pct_min' and '1pct_plu' are confidence bounds, excluded
BLE_TO_AEP = {
    '10-year':  '10P',   # 10% AEP
    '25-year':  '4P',    #  4% AEP  
    '50-year':  '2P',    #  2% AEP
    '100-year': '1P',    #  1% AEP
    '500-year': '0.2P',  #  0.2% AEP
}

# Storm order for capacity analysis (smallest to largest)
STORM_ORDER_BLE = ['10P', '4P', '2P', '1P', '0.2P']

# Segment length for aggregation (0.25 miles = 1320 ft)
SEGMENT_LENGTH = 1320.0

print(f"Reach: {BLE_RIVER} / {BLE_REACH_NAME}")
print(f"Storm order: {STORM_ORDER_BLE}")

In [ ]:
# Download and organize the BLE model using RasEbfeModels
# Auto-downloads 290.7 MB from S3 if not already cached
organized = RasEbfeModels.organize_lower_colorado_cummins(
    river=BLE_RIVER,
    reach=BLE_REACH_NAME
)

# Project folder is under RAS Model/{River}/{Reach}/
project_folder = organized / "RAS Model" / BLE_RIVER / BLE_REACH_NAME

print(f"Organized to: {organized}")
print(f"Project folder: {project_folder}")
print(f"\nProject files:")
for f in sorted(project_folder.iterdir()):
    if f.is_file():
        print(f"  {f.name} ({f.stat().st_size:,} bytes)")

In [ ]:
# Initialize and run with HEC-RAS 6.5 to generate HDF
ras = init_ras_project(project_folder, RAS_VERSION)

print(f"Project: {ras.project_name}")
print(f"Plans: {len(ras.plan_df)}")
print(f"Geometry: {ras.geom_df['geom_title'].iloc[0]}")
print(f"Flow profiles: {ras.flow_df['Flow Title'].iloc[0]}")
print(f"Cross sections: {ras.geom_df['num_cross_sections'].iloc[0]}")

# Run the plan to generate HDF results
print("\nRunning HEC-RAS...")
RasCmdr.compute_plan("01", ras_object=ras)

# Verify HDF created
hdf_files = list(project_folder.glob("*.hdf"))
for h in hdf_files:
    print(f"  {h.name} ({h.stat().st_size:,} bytes)")

## Step 1: Extract Bank Elevations

`extract_bank_elevations()` reads cross section geometry from the geometry HDF
and interpolates elevations at the left and right bank stations. The "controlling"
bank elevation (the higher of the two) determines where overtopping occurs.

In [ ]:
# Get geometry HDF path
geom_hdf = project_folder / f"{ras.project_name}.g01.hdf"

bank_df = HdfChannelCapacity.extract_bank_elevations(str(geom_hdf))

print(f"Cross sections: {len(bank_df)}")
print(f"\nBank elevation range:")
print(f"  Left:  {bank_df['left_bank_elev'].min():.1f} - {bank_df['left_bank_elev'].max():.1f} ft")
print(f"  Right: {bank_df['right_bank_elev'].min():.1f} - {bank_df['right_bank_elev'].max():.1f} ft")
print(f"  Controlling: {bank_df['controlling_bank_elev'].min():.1f} - {bank_df['controlling_bank_elev'].max():.1f} ft")

bank_df.head(10)

In [ ]:
# Plot bank elevations along the channel
fig, ax = plt.subplots(figsize=(14, 5))

rs_float = bank_df['RS'].astype(float)

ax.plot(rs_float, bank_df['left_bank_elev'], 'b-o', markersize=3, label='Left Bank')
ax.plot(rs_float, bank_df['right_bank_elev'], 'r-o', markersize=3, label='Right Bank')
ax.plot(rs_float, bank_df['controlling_bank_elev'], 'k--', linewidth=2, label='Controlling Bank')

ax.set_xlabel('River Station (ft)')
ax.set_ylabel('Elevation (ft)')
ax.set_title(f'{BLE_REACH_NAME} - Bank Elevations')
ax.legend()
ax.invert_xaxis()  # Downstream to right
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 2: Extract Steady-State Profile WSE

`extract_steady_profile_wse()` reads individual steady-state profiles from a single
plan HDF. Unlike `extract_max_wse()` (which collapses profiles to max), this method
preserves each profile as a separate column — essential for capacity analysis.

The BLE model has 7 profiles. We map 5 standard AEP events to the HCFCD storm order.

In [ ]:
# Extract all steady-state profiles from the plan HDF
plan_hdf = project_folder / f"{ras.project_name}.p01.hdf"

all_profiles = HdfChannelCapacity.extract_steady_profile_wse(str(plan_hdf))

print(f"Profiles found: {[c for c in all_profiles.columns if c not in ('River','Reach','RS')]}")
print(f"Cross sections: {len(all_profiles)}")

# Rename BLE profile names to AEP labels and select standard events
wse_df = all_profiles.rename(columns=BLE_TO_AEP)
keep_cols = ['River', 'Reach', 'RS'] + STORM_ORDER_BLE
wse_df = wse_df[keep_cols]

print(f"\nSelected AEP profiles: {STORM_ORDER_BLE}")
print(f"Excluded: 1pct_min, 1pct_plu (confidence bounds)")

wse_df.head(10)

In [ ]:
# Plot WSE profiles with controlling bank elevation
fig, ax = plt.subplots(figsize=(14, 6))

rs_float = wse_df['RS'].astype(float)
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']
labels = ['10-yr (10P)', '25-yr (4P)', '50-yr (2P)', '100-yr (1P)', '500-yr (0.2P)']

for storm, color, label in zip(STORM_ORDER_BLE, colors, labels):
    ax.plot(rs_float, wse_df[storm], color=color, linewidth=1.5, label=label)

# Overlay controlling bank elevation
bank_rs = bank_df['RS'].astype(float)
ax.plot(bank_rs, bank_df['controlling_bank_elev'], 'k--', linewidth=2, label='Controlling Bank')

ax.set_xlabel('River Station (ft)')
ax.set_ylabel('WSE (ft)')
ax.set_title(f'{BLE_REACH_NAME} - Water Surface Profiles vs Bank Elevation')
ax.legend(loc='upper right', fontsize=9)
ax.invert_xaxis()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 3: Determine Channel Capacity

`determine_capacity()` tests storms from smallest (10P) to largest (0.2P) at each
cross section. The capacity level is the largest storm the channel can contain
without the WSE exceeding the controlling bank elevation.

In [ ]:
capacity_df = HdfChannelCapacity.determine_capacity(
    bank_df, wse_df, storm_order=STORM_ORDER_BLE
)

print(f"Cross sections analyzed: {len(capacity_df)}")
print(f"\nCapacity distribution:")
for level in sorted(capacity_df['capacity_level'].unique()):
    count = (capacity_df['capacity_level'] == level).sum()
    category = LEVEL_TO_CATEGORY.get(level, 'Unknown')
    print(f"  Level {level} ({category}): {count} XS")

capacity_df[['RS', 'controlling_bank_elev', 'capacity_level', 
             'capacity_category', 'last_contained_storm']].head(15)

In [ ]:
# Color-coded capacity level along the channel
fig, ax = plt.subplots(figsize=(14, 4))

rs_float = capacity_df['RS'].astype(float)
levels = capacity_df['capacity_level'].values

# Color map: green (good capacity) to red (poor capacity)
capacity_colors = {
    1: '#D32F2F',  # Red - poor
    2: '#F57C00',  # Orange
    3: '#FBC02D',  # Yellow
    4: '#7CB342',  # Light green
    5: '#388E3C',  # Green
    6: '#1B5E20',  # Dark green - excellent
    7: '#B71C1C',  # Dark red - worst
}

for i in range(len(rs_float) - 1):
    color = capacity_colors.get(levels[i], '#888888')
    ax.barh(0, rs_float.iloc[i] - rs_float.iloc[i+1], 
            left=rs_float.iloc[i+1], height=0.8, color=color, edgecolor='white', linewidth=0.5)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=capacity_colors[lv], label=f'Level {lv}: {LEVEL_TO_CATEGORY[lv]}')
    for lv in sorted(capacity_colors.keys()) if lv in levels
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=8, ncol=2)

ax.set_xlabel('River Station (ft)')
ax.set_title(f'{BLE_REACH_NAME} - Channel Capacity by Cross Section')
ax.set_yticks([])
ax.invert_xaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Step 4: Segment Aggregation

`segment_channel()` groups cross sections into fixed-length segments (default: 0.25 mi)
and computes a weighted average capacity for each segment. The capacity is rounded DOWN
(conservative) to ensure the weakest areas in a segment are not masked.

In [ ]:
segments_df = HdfChannelCapacity.segment_channel(
    capacity_df, segment_length=SEGMENT_LENGTH
)

print(f"Segments created: {len(segments_df)}")
print(f"Segment length: {SEGMENT_LENGTH:.0f} ft ({SEGMENT_LENGTH/5280:.2f} mi)")

segments_df

In [ ]:
# Segment capacity bar chart
fig, ax = plt.subplots(figsize=(12, 5))

seg_ids = segments_df['segment_id']
seg_levels = segments_df['capacity_level']
bar_colors = [capacity_colors.get(lv, '#888888') for lv in seg_levels]

bars = ax.bar(seg_ids, seg_levels, color=bar_colors, edgecolor='white', linewidth=0.5)

ax.set_xlabel('Segment ID (upstream to downstream)')
ax.set_ylabel('Capacity Level')
ax.set_title(f'{BLE_REACH_NAME} - Segment Capacity (0.25-mile segments)')
ax.set_ylim(0, 8)
ax.set_yticks(range(1, 8))
ax.set_yticklabels([f'{i}: {LEVEL_TO_CATEGORY[i][:15]}' for i in range(1, 8)], fontsize=8)
ax.axhline(y=4, color='green', linestyle='--', alpha=0.5, label='100-yr capacity (Level 4)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Step 5: System Capacity Summary

`system_capacity_summary()` generates the overall capacity distribution — equivalent
to HCFCD Table 2 in channel capacity reports. Shows what percentage of the channel
length falls into each capacity level.

In [ ]:
summary_df = HdfChannelCapacity.system_capacity_summary(capacity_df)

print(f"Total channel length: {summary_df['channel_length_ft'].sum():,.0f} ft ")
print(f"                    = {summary_df['channel_length_ft'].sum()/5280:.2f} mi")
print()
summary_df

In [ ]:
# Capacity distribution pie chart
nonzero = summary_df[summary_df['channel_length_ft'] > 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
pie_colors = [capacity_colors.get(lv, '#888888') for lv in nonzero['capacity_level']]
wedges, texts, autotexts = ax1.pie(
    nonzero['percent_of_total'], 
    labels=[f"Level {row['capacity_level']}" for _, row in nonzero.iterrows()],
    colors=pie_colors,
    autopct='%1.1f%%',
    startangle=90
)
ax1.set_title('Capacity Distribution by Channel Length')

# Bar chart
ax2.barh(
    [f"Level {row['capacity_level']}: {row['capacity_category']}" for _, row in nonzero.iterrows()],
    nonzero['channel_length_ft'] / 5280,  # Convert to miles
    color=pie_colors
)
ax2.set_xlabel('Channel Length (mi)')
ax2.set_title('Channel Length by Capacity Level')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## Step 6: Full Pipeline (One Call)

`analyze_channel_capacity()` orchestrates all steps in a single call. For BLE models,
we first extract profiles with `extract_steady_profile_wse()` and pass the resulting
DataFrame as a pre-built WSE table.

In [ ]:
# For the full pipeline, we pass the plan HDF as a single-element list.
# Since BLE uses a single plan with multiple steady profiles,
# we use extract_steady_profile_wse + determine_capacity directly.
#
# The full pipeline (analyze_channel_capacity) is designed for the case where
# each AEP is a separate plan HDF. Let's verify our step-by-step results match.

# Re-run step by step to confirm
bank_check = HdfChannelCapacity.extract_bank_elevations(str(geom_hdf))
wse_check = HdfChannelCapacity.extract_steady_profile_wse(str(plan_hdf))
wse_check = wse_check.rename(columns=BLE_TO_AEP)
wse_check = wse_check[['River', 'Reach', 'RS'] + STORM_ORDER_BLE]

cap_check = HdfChannelCapacity.determine_capacity(bank_check, wse_check, storm_order=STORM_ORDER_BLE)
summary_check = HdfChannelCapacity.system_capacity_summary(cap_check)

# Verify match
match = (cap_check['capacity_level'].values == capacity_df['capacity_level'].values).all()
print(f"Step-by-step results match: {match}")

# Show summary comparison
print(f"\nSummary verification:")
for _, row in summary_check.iterrows():
    if row['channel_length_ft'] > 0:
        print(f"  Level {row['capacity_level']}: {row['percent_of_total']:.1f}% ({row['xs_count']} XS)")

## Step 7: Compare Conditions

`compare_conditions()` compares two sets of capacity results to evaluate the impact
of proposed improvements. Here we simulate raising all bank elevations by 2 ft to
demonstrate the comparison workflow.

In [ ]:
# Simulate proposed condition: raise banks by 2 ft
proposed_banks = bank_df.copy()
proposed_banks['left_bank_elev'] += 2.0
proposed_banks['right_bank_elev'] += 2.0
proposed_banks['controlling_bank_elev'] += 2.0

# Determine capacity for proposed condition
proposed_capacity = HdfChannelCapacity.determine_capacity(
    proposed_banks, wse_df, storm_order=STORM_ORDER_BLE
)
proposed_summary = HdfChannelCapacity.system_capacity_summary(proposed_capacity)

# Build results dicts for compare_conditions
existing_results = {
    'xs_capacity': capacity_df,
    'segments': segments_df,
    'summary': summary_df,
    'bank_elevations': bank_df,
    'max_wse': wse_df,
}

proposed_segments = HdfChannelCapacity.segment_channel(proposed_capacity, SEGMENT_LENGTH)
proposed_results = {
    'xs_capacity': proposed_capacity,
    'segments': proposed_segments,
    'summary': proposed_summary,
    'bank_elevations': proposed_banks,
    'max_wse': wse_df,
}

comparison = HdfChannelCapacity.compare_conditions(existing_results, proposed_results)

n_improved = comparison['improved'].sum()
n_degraded = (comparison['change'] < 0).sum()
n_unchanged = (comparison['change'] == 0).sum()

print(f"Comparison results (+2 ft bank raise):")
print(f"  Improved:  {n_improved} XS")
print(f"  Unchanged: {n_unchanged} XS")
print(f"  Degraded:  {n_degraded} XS")

comparison[['RS', 'existing_level', 'existing_category', 
            'proposed_level', 'proposed_category', 'change']].head(15)

In [ ]:
# Existing vs Proposed capacity comparison
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

rs_float = comparison['RS'].astype(float)

# Existing
ax1.bar(rs_float, comparison['existing_level'], width=200, 
        color=[capacity_colors.get(lv, '#888') for lv in comparison['existing_level']])
ax1.set_ylabel('Capacity Level')
ax1.set_title('Existing Conditions')
ax1.set_ylim(0, 8)
ax1.grid(True, alpha=0.3)

# Proposed
ax2.bar(rs_float, comparison['proposed_level'], width=200,
        color=[capacity_colors.get(lv, '#888') for lv in comparison['proposed_level']])
ax2.set_ylabel('Capacity Level')
ax2.set_xlabel('River Station (ft)')
ax2.set_title('Proposed Conditions (+2 ft bank raise)')
ax2.set_ylim(0, 8)
ax2.invert_xaxis()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Key Takeaways

1. **`extract_bank_elevations()`** reads bank station elevations from the geometry HDF using
   `np.interp()` on station-elevation data.

2. **`extract_steady_profile_wse()`** extracts individual steady-state profiles from a single
   plan HDF, preserving each AEP event as a separate column.

3. **`determine_capacity()`** tests storms from smallest to largest, classifying each cross
   section into capacity levels 1-7.

4. **`segment_channel()`** aggregates XS-level capacity into fixed-length segments with
   conservative (FLOOR) weighting.

5. **`system_capacity_summary()`** produces the HCFCD Table 2 capacity distribution.

6. **`compare_conditions()`** enables before/after analysis for improvement projects.

**For BLE models** with a single plan and multiple steady profiles, use
`extract_steady_profile_wse()` instead of `extract_max_wse()` (which is designed
for multi-plan workflows where each AEP is a separate plan HDF).

**References:**
- HCFCD Policy, Criteria, and Procedure Manual (PCPM)
- FEMA BLE program: https://www.fema.gov/flood-maps/products-tools/base-level-engineering